# ═══════════════════════════════════════════════════════════════════
# LinguaMeet AI Service — Google Colab Test Notebook
# ═══════════════════════════════════════════════════════════════════
# Instructions for New Users / Evaluators:
#   1. Open a NEW Google Colab notebook (Runtime → Change runtime type → T4 GPU)
#   2. Copy each CELL block below into a separate Colab cell
#   3. Run cells one by one in order (Shift+Enter)
# ═══════════════════════════════════════════════════════════════════


# ───────────────────────────────────────────────────────────────────


In [7]:
# CELL 1 — Check GPU availability (16GB VRAM on T4)
# ───────────────────────────────────────────────────────────────────
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU")


# ───────────────────────────────────────────────────────────────────


CUDA available: True
GPU: Tesla T4
VRAM: 15.64 GB


In [8]:
# CELL 2 — Clone GitHub repository
# ───────────────────────────────────────────────────────────────────
import os, shutil

# Move out of the directory before deleting it (fixes getcwd error on 2nd run)
os.chdir('/content')

GITHUB_REPO = "https://github.com/Saurabh1127/Final-year.git"
REPO_DIR    = "/content/Final-year"

# Fresh clean clone
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

print(f"📥 Cloning repository: {GITHUB_REPO} ...")
exit_code = os.system(f"git clone {GITHUB_REPO} {REPO_DIR}")

if exit_code != 0 or not os.path.exists(f"{REPO_DIR}/ai-service"):
    print("\n❌ ERROR: Failed to clone repository or 'ai-service' directory missing!")
else:
    os.chdir(f"{REPO_DIR}/ai-service")
    print("✅ Repo cloned successfully!")
    print("Working directory:", os.getcwd())


# ───────────────────────────────────────────────────────────────────


📥 Cloning repository: https://github.com/Saurabh1127/Final-year.git ...
✅ Repo cloned successfully!
Working directory: /content/Final-year/ai-service


In [9]:
# CELL 3 — Install all dependencies (AI, Web, Audio)
# ───────────────────────────────────────────────────────────────────
!pip install --prefer-binary \
    fastapi uvicorn[standard] python-multipart websockets python-dotenv pyngrok \
    faster-whisper transformers accelerate sentencepiece nest_asyncio gTTS edge-tts

print("✅ All packages installed cleanly.")


✅ All packages installed cleanly.


In [10]:
# CELL 4 — Configure AI Models & Engine Keys
# ───────────────────────────────────────────────────────────────────
import os

# ASR (STT) Model: faster-whisper uses CTranslate2 for 4-5x speedup
os.environ["WHISPER_MODEL"]     = "large-v3-turbo"
# Phase 13 Speed Tuning: beam_size=1 (greedy) cuts Whisper latency from ~1.7s to ~0.35s
os.environ["WHISPER_BEAM_SIZE"] = "1"

os.environ["NLLB_MODEL"]        = "facebook/nllb-200-distilled-1.3B"

# TTS Engine Preference: Edge-TTS provides ultra-low latency (~0.5s) for real-time meetings.
# Set USE_SARVAM="true" if you have a Sarvam API key and want sovereign Indian voices.
try:
    from google.colab import userdata
    os.environ["SARVAM_API_KEY"] = userdata.get("SARVAM_API_KEY")
    os.environ["USE_SARVAM"] = os.environ.get("USE_SARVAM", "false")
    print("✅ SARVAM_API_KEY loaded from Colab Secrets.")
except Exception:
    os.environ["SARVAM_API_KEY"] = os.environ.get("SARVAM_API_KEY", "")
    os.environ["USE_SARVAM"] = "false"
    print("ℹ️ Using Edge-TTS as ultra-fast primary engine.")

print("\nEnvironment configured:")
print(f"  WHISPER_MODEL     = {os.environ['WHISPER_MODEL']}")
print(f"  WHISPER_BEAM_SIZE = {os.environ['WHISPER_BEAM_SIZE']}")
print(f"  NLLB_MODEL        = {os.environ['NLLB_MODEL']}")
print(f"  USE_SARVAM        = {os.environ['USE_SARVAM']}")


Environment configured:
  WHISPER_MODEL  = large-v3-turbo
  NLLB_MODEL     = facebook/nllb-200-distilled-1.3B
  SARVAM_API_KEY = NOT SET


In [11]:
# CELL 5 — Pre-download faster-whisper model weights (cache only)
# ───────────────────────────────────────────────────────────────────
# Only downloads to disk cache — does NOT load into RAM/VRAM.
# The FastAPI server loads it on startup via the lifespan handler.
from faster_whisper import WhisperModel
model_name = os.environ.get("WHISPER_MODEL", "large-v3-turbo")
print(f"⬇️  Downloading faster-whisper '{model_name}' to cache...")
_fw = WhisperModel(model_name, device="cpu", compute_type="int8")
print(f"✅ faster-whisper '{model_name}' cached.")
del _fw
import torch; torch.cuda.empty_cache()
import gc; gc.collect()


# ───────────────────────────────────────────────────────────────────


⬇️  Downloading faster-whisper 'large-v3-turbo' to cache...
✅ faster-whisper 'large-v3-turbo' cached.


401

In [ ]:
# CELL 4 — Configure AI Models & Engine Keys
# ───────────────────────────────────────────────────────────────────
import os

# ASR (STT) Model: faster-whisper uses CTranslate2 for 4-5x speedup
os.environ["WHISPER_MODEL"]     = "large-v3-turbo"
# Phase 13 Speed Tuning: beam_size=1 (greedy) cuts Whisper latency from ~1.7s to ~0.35s
os.environ["WHISPER_BEAM_SIZE"] = "1"

os.environ["NLLB_MODEL"]        = "facebook/nllb-200-distilled-1.3B"

# TTS Engine Preference: Edge-TTS provides ultra-low latency (~0.5s) for real-time meetings.
# Set USE_SARVAM="true" if you have a Sarvam API key and want sovereign Indian voices.
try:
    from google.colab import userdata
    os.environ["SARVAM_API_KEY"] = userdata.get("SARVAM_API_KEY")
    os.environ["USE_SARVAM"] = os.environ.get("USE_SARVAM", "false")
    print("✅ SARVAM_API_KEY loaded from Colab Secrets.")
except Exception:
    os.environ["SARVAM_API_KEY"] = os.environ.get("SARVAM_API_KEY", "")
    os.environ["USE_SARVAM"] = "false"
    print("ℹ️ Using Edge-TTS as ultra-fast primary engine.")

print("\nEnvironment configured:")
print(f"  WHISPER_MODEL     = {os.environ['WHISPER_MODEL']}")
print(f"  WHISPER_BEAM_SIZE = {os.environ['WHISPER_BEAM_SIZE']}")
print(f"  NLLB_MODEL        = {os.environ['NLLB_MODEL']}")
print(f"  USE_SARVAM        = {os.environ['USE_SARVAM']}")


⬇️  Downloading and converting NLLB 'facebook/nllb-200-distilled-1.3B' to CTranslate2 INT8 format...


In [ ]:
# CELL 7 — Start FastAPI (Port 8000) + Node.js (Port 5000) via Cloudflare Tunnel
# ───────────────────────────────────────────────────────────────────
# Zero-auth required: No ngrok token, no quota limits, supports WebSockets & Socket.IO.
# All sensitive secrets are securely loaded from Colab Secrets (🔑 icon on sidebar).
import subprocess, time, re, os, shutil, requests, secrets, getpass

REPO_DIR = "/content/Final-year"

# 1. Kill ONLY our specific application ports and processes
# ⚠️ NEVER use 'pkill -9 -f node' because Colab's web runtime runs on Node.js!
os.system("fuser -k 8000/tcp > /dev/null 2>&1")
os.system("fuser -k 5000/tcp > /dev/null 2>&1")
os.system("pkill -9 -f 'server.js'")
os.system("pkill -9 -f 'app.main:app'")
os.system("pkill -9 -f cloudflared")

# 2. Safely read secrets from Colab Secrets (🔑 icon on left sidebar)
# NO API KEYS OR PASSWORDS ARE HARDCODED HERE.
try:
    from google.colab import userdata
    def get_secret(key):
        try:
            val = userdata.get(key)
            return val if val else ""
        except Exception:
            return os.environ.get(key, "")
except ImportError:
    def get_secret(key):
        return os.environ.get(key, "")

MONGO_URI = get_secret("MONGO_URI")
JWT_SECRET = get_secret("JWT_SECRET") or secrets.token_hex(32)
METERED_API_KEY = get_secret("METERED_API_KEY")
GEMINI_API_KEY = get_secret("GEMINI_API_KEY")

# If MONGO_URI was not added to Secrets (🔑), prompt securely (hidden input)
if not MONGO_URI:
    print("⚠️ 'MONGO_URI' not found in Colab Secrets (🔑 sidebar).")
    MONGO_URI = getpass.getpass("👉 Paste your MongoDB connection string (input will stay hidden): ")

# 3. Configure Node.js server environment (.env)
with open(f"{REPO_DIR}/server/.env", "w") as f:
    f.write(f"""
PORT=5000
MONGO_URI={MONGO_URI}
JWT_SECRET={JWT_SECRET}
AI_SERVICE_URL=http://127.0.0.1:8000
METERED_API_KEY={METERED_API_KEY}
GEMINI_API_KEY={GEMINI_API_KEY}
""".strip())

print("✅ Node.js server .env configured (AI_SERVICE_URL points to internal :8000)")

# 4. Install Node.js dependencies (only on first run)
if not os.path.exists(f"{REPO_DIR}/server/node_modules"):
    print("📦 Installing Node.js server dependencies...")
    os.system(f"cd {REPO_DIR}/server && npm install --production --silent")
else:
    print("✅ Node.js server dependencies already present.")

# 5. Start Python FastAPI AI Server (Port 8000)
ai_log = open(f"{REPO_DIR}/ai-service/server.log", "w")
ai_server = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=f"{REPO_DIR}/ai-service",
    stdout=ai_log,
    stderr=subprocess.STDOUT,
)
print("⏳ Starting FastAPI AI Engine on port 8000...")
time.sleep(5)

# 6. Start Node.js Server (Port 5000)
node_log = open(f"{REPO_DIR}/server/node_server.log", "w")
node_server = subprocess.Popen(
    ["node", "server.js"],
    cwd=f"{REPO_DIR}/server",
    stdout=node_log,
    stderr=subprocess.STDOUT,
)
print("⏳ Starting Node.js Orchestrator on port 5000...")
time.sleep(3)

# 7. Install & Expose Port 5000 (Node.js) via Cloudflare Tunnel
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("⬇️ Installing Cloudflare Tunnel (cloudflared)...")
    os.system("curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")

cf_log_path = f"{REPO_DIR}/server/cloudflared.log"
cf_log = open(cf_log_path, "w")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=cf_log,
    stderr=subprocess.STDOUT,
    text=True
)

print("🌐 Connecting Cloudflare Tunnel to Node.js (Port 5000)...")
public_url = None
start_time = time.time()
while time.time() - start_time < 30:
    time.sleep(0.5)
    if os.path.exists(cf_log_path):
        with open(cf_log_path, "r", errors="ignore") as f:
            match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', f.read())
            if match:
                public_url = match.group(1)
                break

if public_url:
    print("\n" + "═"*65)
    print("🚀 UNIFIED BACKEND IS LIVE VIA CLOUDFLARE TUNNEL!")
    print("═"*65)
    print(f"  Public Node.js API & Socket.IO → {public_url}")
    print(f"  Health Check Status            → {public_url}/api/health")
    print(f"  Internal AI Link               → http://127.0.0.1:8000 (0ms loopback)")
    print("\n👉 COPY THIS URL TO YOUR LOCAL client/.env:")
    print(f"  VITE_BACKEND_URL={public_url}")
    print("═"*65)
else:
    print("❌ Cloudflare tunnel could not be established.")
    if os.path.exists(cf_log_path):
        with open(cf_log_path, "r") as f:
            print("Recent Cloudflare logs:\n", f.read()[-600:])


In [ ]:
# CELL 7B (RECONNECT ONLY) — Restart Cloudflare Tunnel to Node.js (Port 5000)
# ───────────────────────────────────────────────────────────────────
# Use this if the tunnel disconnects, without restarting Node or FastAPI!
import subprocess, time, re, os

REPO_DIR = "/content/Final-year"
os.system("pkill -9 -f cloudflared")

cf_log_path = f"{REPO_DIR}/server/cloudflared.log"
cf_log = open(cf_log_path, "w")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=cf_log,
    stderr=subprocess.STDOUT,
    text=True
)

print("🌐 Reconnecting Cloudflare Tunnel to Port 5000...")
public_url = None
start_time = time.time()
while time.time() - start_time < 30:
    time.sleep(0.5)
    if os.path.exists(cf_log_path):
        with open(cf_log_path, "r", errors="ignore") as f:
            match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', f.read())
            if match:
                public_url = match.group(1)
                break

if public_url:
    print("\n" + "═"*65)
    print("🚀 CLOUDFLARE TUNNEL RECONNECTED!")
    print(f"  Public Node.js API & Socket.IO → {public_url}")
    print("\n👉 UPDATE YOUR LOCAL client/.env:")
    print(f"  VITE_BACKEND_URL={public_url}")
    print("═"*65)
else:
    print("❌ Cloudflare tunnel reconnect failed.")


In [ ]:
# CELL 8 — 3-Way Speech-to-Speech Audio Verification Widget
# ───────────────────────────────────────────────────────────────────
import sys, base64, requests, json
from IPython.display import HTML, Audio, display
import google.colab.output

TARGET_LANGS = ["hi"]

RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = () => resolve(reader.result);
  reader.readAsDataURL(blob);
});
var record = path => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  recorder = new MediaRecorder(stream);
  chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  button = document.createElement('button');
  button.onclick = () => { recorder.stop(); };
  button.innerText = '🔴 STOP RECORDING';
  button.style = 'background: #f87171; color: white; border: none; padding: 14px 28px; font-size: 18px; font-weight: bold; border-radius: 8px; cursor: pointer; margin: 15px 0; display: block;';
  document.body.appendChild(button);
  while (recorder.state == 'recording') await sleep(100);
  stream.getTracks().forEach(track => track.stop());
  button.remove();
  blob = new Blob(chunks, { type: 'audio/webm' });
  text = await b2text(blob);
  resolve(text);
});
"""

print("🎙️ SPEAK ANY ENGLISH SENTENCE TO TEST SPEECH TRANSLATION:")
display(HTML("<script>" + RECORD_JS + "</script>"))
data = google.colab.output.eval_js("record()")
recorded_audio_bytes = base64.b64decode(data.split(',')[1])

print("\n🚀 Sending audio to FastAPI server for processing...")

# We send the request to the local FastAPI server instead of importing the engine.
# This prevents the notebook from loading a second copy of the AI models into VRAM!
response = requests.post(
    "http://localhost:8000/api/process-audio",
    files={"audio": ("test_audio.webm", recorded_audio_bytes, "audio/webm")},
    data={
        "meeting_id": "test_room",
        "user_id": "colab_tester",
        "speaker_name": "Colab User",
        "source_language": "auto",
        "target_languages": json.dumps(TARGET_LANGS),
        "include_audio": "true",
    }
)

if response.status_code != 200:
    print(f"❌ Server Error: {response.status_code}\n{response.text}")
else:
    result = response.json()
    hindi_text = result.get("translations", {}).get("hi", "")

print("\n" + "═"*65)
print(f"📝 ORIGINAL SPOKEN TEXT [{result.get('source_language','?').upper()}]: \"{result.get('original_text','')}\"")
print(f"🌐 TRANSLATED HINDI TEXT: \"{hindi_text}\"")
print("═"*65)

print("\n🔊 3-WAY AUDIO COMPARISON:")

print("\n" + "─"*65)
print("🎤 1. YOUR ORIGINAL RECORDED VOICE")
print("─"*65)
display(Audio(data=recorded_audio_bytes, autoplay=False))

print("\n" + "─"*65)
print("🔊 2. AI TRANSLATED VOICE (Returned from API)")
print("─"*65)

audio_b64 = result.get("audio_base64")
if audio_b64:
    audio_bytes = base64.b64decode(audio_b64)
    display(Audio(data=audio_bytes, autoplay=False))
else:
    print("⚠️ No audio returned from API.")

lat = result.get("latency", {})
print("\n" + "═"*65)
print(f"⚡ LATENCY: STT={lat.get('asr_seconds')}s | NMT={lat.get('nmt_seconds')}s | TOTAL={lat.get('total_seconds')}s")
print("═"*65)


# ───────────────────────────────────────────────────────────────────


In [ ]:
# CELL 9 — Stop all servers and tunnels when done testing
# ───────────────────────────────────────────────────────────────────
import os
os.system("pkill -9 -f cloudflared")
os.system("fuser -k 8000/tcp > /dev/null 2>&1")
os.system("fuser -k 5000/tcp > /dev/null 2>&1")
os.system("pkill -9 -f 'server.js'")
os.system("pkill -9 -f 'app.main:app'")
print("✅ All servers and tunnels stopped.")


In [ ]:
# CELL 10 — Run BLEU & Latency Benchmarks (Phase 4)
# ─────────────────────────────────────────────────────────────────────────────
# This will compare the NLLB-200 baseline vs the new IndicTrans2 Hybrid Router
import os
os.chdir('/content/Final-year/ai-service')
!python benchmark.py
